<a href="https://colab.research.google.com/github/Joao-Eduardo-Hiku/AnimalInformationVisionAI/blob/master/Simuldor_Ahmes_Trabalho_de_Arquitetura_de_Computadores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulador Ahmes - Arquitetura de Computadores

**Grupo de Trabalho**
* Integrante 1: Geovane Stuart Barroso
* Integrante 2: Gustavo Amaral Dias
* Integrante 3: João Eduardo
* Integrante 4: Renato Maciel Silva

## Resumo da Arquitetura Ahmes
O Ahmes é um processador hipotético de 8 bits com espaço de endereçamento de 256 bytes. Possui um acumulador (`AC`) de 8 bits, um contador de programa (`PC`) de 8 bits e cinco flags de estado: Negativo (`N`), Zero (`Z`), Vai-um (`C`), Estouro (`V`) e Meio-byte (`B`).

## 1. Estrutura Base e Ciclo de Execução (Core)


**Escopo:**
- Inicialização da CPU (Memória de 256 bytes, Registradores AC e PC, Flags N, Z, C, V, B).
- Métodos de controle: reset de estado, carregamento do programa na memória e controle do ciclo Busca-Decodificação-Execução (`step`).

In [37]:
class AhmesCore:
    # Mapeamento: opcode -> (mnemonico, tamanho_em_bytes)
    OPCODES = {
        0x00: ("NOP", 1), 0x10: ("STA", 2), 0x20: ("LDA", 2), 0x30: ("ADD", 2),
        0x40: ("OR",  2), 0x50: ("AND", 2), 0x60: ("NOT", 1), 0x70: ("SUB", 2),
        0x80: ("JMP", 2), 0x90: ("JN",  2), 0x94: ("JP",  2), 0x98: ("JV",  2),
        0x9C: ("JNV", 2), 0xA0: ("JZ",  2), 0xA4: ("JNZ", 2), 0xB0: ("JC",  2),
        0xB4: ("JNC", 2), 0xB8: ("JB",  2), 0xBC: ("JNB", 2), 0xE0: ("SHR", 1),
        0xE1: ("SHL", 1), 0xE2: ("ROR", 1), 0xE3: ("ROL", 1), 0xF0: ("HLT", 1),
    }

    # Opcodes reconhecidos só pelos 4 bits mais significativos (bits 3-0 ignorados).
    # Saltos (0x9_, 0xA_, 0xB_) e deslocamentos (0xE_) exigem o byte exato.
    NIBBLE_ALTO = (0x00, 0x10, 0x20, 0x30, 0x40, 0x50, 0x60, 0x70, 0x80, 0xF0)

    FLAGS = ("N", "Z", "C", "V", "B")

    def __init__(self, ula_executor=None):
        self.memoria = [0] * 256
        self.ac = 0
        self.pc = 0
        self.flags = self._flags_iniciais()
        self.halted = False
        self.ciclos = 0
        self.ula_executor = ula_executor  # Função da Pessoa 2

    @staticmethod
    def _flags_iniciais():
        # AC = 0 no início, logo Z = 1. Demais flags zeradas.
        return {"N": False, "Z": True, "C": False, "V": False, "B": False}

    def reset(self):
        """Zera AC, PC, flags e contador de ciclos. Não altera a memória."""
        self.ac = 0
        self.pc = 0
        self.flags = self._flags_iniciais()
        self.halted = False
        self.ciclos = 0

    def limpar_memoria(self):
        self.memoria = [0] * 256

    def carregar_programa(self, bytes_programa, endereco_inicial=0):
        if endereco_inicial + len(bytes_programa) > 256:
            raise ValueError("Programa excede 256 bytes.")
        for i, byte in enumerate(bytes_programa):
            self.memoria[endereco_inicial + i] = byte & 0xFF

    def carregar_mem_txt(self, caminho):

        with open(caminho, "r", encoding="utf-8") as f:
            valores = [linha.strip() for linha in f if linha.strip()]
        if len(valores) > 256:
            raise ValueError(f"{caminho}: {len(valores)} valores, máximo 256.")
        novos = []
        for n, texto in enumerate(valores, start=1):
            try:
                v = int(texto)
            except ValueError:
                raise ValueError(f"{caminho}, linha {n}: valor inválido '{texto}'.")
            if not (0 <= v <= 255):
                raise ValueError(f"{caminho}, linha {n}: valor {v} fora de 0..255.")
            novos.append(v)
        self.limpar_memoria()
        self.carregar_programa(novos)
        return len(novos)

    def salvar_mem_txt(self, caminho):
        """Grava a memória atual no formato do mem.txt (um decimal por linha)."""
        with open(caminho, "w", encoding="utf-8") as f:
            for valor in self.memoria:
                f.write(f"{valor}\n")

    def ler_memoria(self, endereco):
        return self.memoria[endereco & 0xFF]

    def escrever_memoria(self, endereco, valor):
        self.memoria[endereco & 0xFF] = valor & 0xFF

    def buscar_instrucao(self):
        instrucao_byte = self.ler_memoria(self.pc)
        self.pc = (self.pc + 1) & 0xFF
        return instrucao_byte

    def decodificar_instrucao(self, instrucao_byte):
        # 1) correspondência exata; 2) nibble alto, só para as instruções que o usam
        if instrucao_byte in self.OPCODES:
            mnemonico, tamanho = self.OPCODES[instrucao_byte]
        elif (instrucao_byte & 0xF0) in self.NIBBLE_ALTO:
            mnemonico, tamanho = self.OPCODES[instrucao_byte & 0xF0]
        else:
            raise ValueError(
                f"Opcode inválido: 0x{instrucao_byte:02X} no endereço {(self.pc - 1) & 0xFF}"
            )

        operando = self.buscar_instrucao() if tamanho == 2 else None
        return mnemonico, operando

    def delegar_execucao(self, mnemonico, operando):
        if mnemonico == "NOP":
            return
        if mnemonico == "HLT":
            self.halted = True
            return
        if self.ula_executor is None:
            raise NotImplementedError(f"ULA não conectada para '{mnemonico}'.")
        self.ula_executor(self, mnemonico, operando)

    def executar_passo(self):
        if self.halted:
            return False
        self.ciclos += 1
        byte_instrucao = self.buscar_instrucao()
        mnemonico, operando = self.decodificar_instrucao(byte_instrucao)
        self.delegar_execucao(mnemonico, operando)
        return not self.halted

    def executar_ate_parar(self, max_ciclos=10000):
        while not self.halted:
            if self.ciclos >= max_ciclos:
                raise RuntimeError("Limite de ciclos atingido (possível loop infinito).")
            self.executar_passo()

    def sinal_ac(self):

        negativo = "Negativo" if self.ac > 127 else "Positivo"
        zero = "Zero" if self.ac == 0 else "Diferente de Zero"
        return negativo, zero

    def estado_atual(self):
        negativo, zero = self.sinal_ac()
        return {
            "ciclo": self.ciclos,
            "PC": f"0x{self.pc:02X} ({self.pc})",
            "AC": f"0x{self.ac:02X} ({self.ac})",
            "Flags": {k: int(self.flags.get(k, False)) for k in self.FLAGS},
            "Sinal": negativo,
            "Zero": zero,
            "Halted": self.halted,
        }


## 2. Conjunto de Instruções e ULA

**Escopo:**
- Lógica de execução de cada instrução do Ahmes.
- Atualização das flags de status baseada no resultado das operações aritméticas, lógicas e de deslocamento.

In [38]:
def atualizar_flags(cpu, resultado_previo, carry=None, overflow=None, borrow=None):
    """Atualiza N e Z a partir do resultado (8 bits). C, V e B só mudam se informados."""
    r = resultado_previo & 0xFF
    cpu.flags["N"] = bool(r & 0x80)
    cpu.flags["Z"] = (r == 0)
    if carry is not None:
        cpu.flags["C"] = bool(carry)
    if overflow is not None:
        cpu.flags["V"] = bool(overflow)
    if borrow is not None:
        cpu.flags["B"] = bool(borrow)


def _ula_add(cpu, op):
    a, b = cpu.ac, cpu.ler_memoria(op)
    bruto = a + b
    r = bruto & 0xFF
    cpu.ac = r
    atualizar_flags(cpu, r, carry=bruto > 0xFF, overflow=bool((a ^ r) & (b ^ r) & 0x80))


def _ula_sub(cpu, op):
    a, b = cpu.ac, cpu.ler_memoria(op)
    r = (a - b) & 0xFF
    cpu.ac = r
    atualizar_flags(cpu, r, overflow=bool((a ^ b) & (a ^ r) & 0x80), borrow=a < b)


def _ula_lda(cpu, op):
    cpu.ac = cpu.ler_memoria(op)
    atualizar_flags(cpu, cpu.ac)


def _ula_sta(cpu, op):
    cpu.escrever_memoria(op, cpu.ac)      # STA não altera flags


def _ula_or(cpu, op):
    cpu.ac = (cpu.ac | cpu.ler_memoria(op)) & 0xFF
    atualizar_flags(cpu, cpu.ac)


def _ula_and(cpu, op):
    cpu.ac = (cpu.ac & cpu.ler_memoria(op)) & 0xFF
    atualizar_flags(cpu, cpu.ac)


def _ula_not(cpu, op):
    cpu.ac = (~cpu.ac) & 0xFF
    atualizar_flags(cpu, cpu.ac)


def _ula_shr(cpu, op):
    c = cpu.ac & 1
    cpu.ac = cpu.ac >> 1
    atualizar_flags(cpu, cpu.ac, carry=c)


def _ula_shl(cpu, op):
    c = (cpu.ac >> 7) & 1
    cpu.ac = (cpu.ac << 1) & 0xFF
    atualizar_flags(cpu, cpu.ac, carry=c)


def _ula_ror(cpu, op):                    # rotação através do carry
    c_antigo, c_novo = int(cpu.flags["C"]), cpu.ac & 1
    cpu.ac = (cpu.ac >> 1) | (c_antigo << 7)
    atualizar_flags(cpu, cpu.ac, carry=c_novo)


def _ula_rol(cpu, op):
    c_antigo, c_novo = int(cpu.flags["C"]), (cpu.ac >> 7) & 1
    cpu.ac = ((cpu.ac << 1) | c_antigo) & 0xFF
    atualizar_flags(cpu, cpu.ac, carry=c_novo)


def _salto(condicao):
    def _f(cpu, op):
        if condicao(cpu.flags):
            cpu.pc = op & 0xFF
    return _f


_TABELA_ULA = {
    "LDA": _ula_lda, "STA": _ula_sta, "ADD": _ula_add, "SUB": _ula_sub,
    "OR": _ula_or, "AND": _ula_and, "NOT": _ula_not,
    "SHR": _ula_shr, "SHL": _ula_shl, "ROR": _ula_ror, "ROL": _ula_rol,
    "JMP": _salto(lambda f: True),
    "JN":  _salto(lambda f: f["N"]),     "JP":  _salto(lambda f: not f["N"]),
    "JV":  _salto(lambda f: f["V"]),     "JNV": _salto(lambda f: not f["V"]),
    "JZ":  _salto(lambda f: f["Z"]),     "JNZ": _salto(lambda f: not f["Z"]),
    "JC":  _salto(lambda f: f["C"]),     "JNC": _salto(lambda f: not f["C"]),
    "JB":  _salto(lambda f: f["B"]),     "JNB": _salto(lambda f: not f["B"]),
    "NOP": lambda cpu, op: None,
    "HLT": lambda cpu, op: setattr(cpu, "halted", True),
}


def executar_ula(cpu, mnemonico, operando=None):
    try:
        instrucao = _TABELA_ULA[mnemonico.upper()]
    except KeyError:
        raise ValueError(f"Mnemônico Ahmes inválido: {mnemonico!r}")
    instrucao(cpu, 0 if operando is None else operando)


def executar_instrucao(cpu, opcode, operando=None):
    mnemonico, _ = cpu.OPCODES[opcode if opcode in cpu.OPCODES else opcode & 0xF0]
    executar_ula(cpu, mnemonico, operando)

ULA_EM_USO = executar_ula


## 3. Montador / Tradutor (Assembler)

**Escopo:**
- Tradução do código fonte em formato de texto Assembly Ahmes para código de máquina (bytes).
- Mapeamento de rótulos/labels para permitir desvios amigáveis.

In [39]:
import os
import re
from typing import Dict, List, Optional, Set, Tuple

class ErroDeMontagem(Exception):

    pass

class MontadorAhmes:

    TAMANHO_MEMORIA = 256

    # Mnemônico -> (Opcode, Tamanho em Bytes)
    TABELA_INSTRUCOES: Dict[str, Tuple[int, int]] = {
        # Instruções de 1 byte
        "NOP": (0x00, 1),   # 0
        "NOT": (0x60, 1),   # 96
        "SHR": (0xE0, 1),   # 224
        "SHL": (0xE1, 1),   # 225
        "ROR": (0xE2, 1),   # 226
        "ROL": (0xE3, 1),   # 227
        "HLT": (0xF0, 1),   # 240

        # Instruções de 2 bytes
        "STA": (0x10, 2),   # 16
        "LDA": (0x20, 2),   # 32
        "ADD": (0x30, 2),   # 48
        "OR":  (0x40, 2),   # 64
        "AND": (0x50, 2),   # 80
        "SUB": (0x70, 2),   # 112
        "JMP": (0x80, 2),   # 128

        # Saltos condicionais
        "JN":  (0x90, 2),   # 144 - N=1
        "JP":  (0x94, 2),   # 148 - N=0
        "JV":  (0x98, 2),   # 152 - V=1
        "JNV": (0x9C, 2),   # 156 - V=0
        "JZ":  (0xA0, 2),   # 160 - Z=1
        "JNZ": (0xA4, 2),   # 164 - Z=0
        "JC":  (0xB0, 2),   # 176 - C=1
        "JNC": (0xB4, 2),   # 180 - C=0
        "JB":  (0xB8, 2),   # 184 - B=1
        "JNB": (0xBC, 2),   # 188 - B=0
    }

    DIRETIVAS = ("ORG", "DB", "EQU")

    OPCODE_PARA_MNEMONICO: Dict[int, str] = {op: m for m, (op, _) in TABELA_INSTRUCOES.items()}

    def __init__(self):
        self.memoria: List[int] = [0] * self.TAMANHO_MEMORIA
        self.tabela_simbolos: Dict[str, int] = {}
        self.log_montagem: List[str] = []
        self._escritos: Set[int] = set()

    def resetar(self) -> None:
        self.memoria = [0] * self.TAMANHO_MEMORIA
        self.tabela_simbolos.clear()
        self.log_montagem.clear()
        self._escritos.clear()

    # utilidades
    @staticmethod
    def _limpar_linha(linha: str) -> str:
        linha = re.split(r"[;#]|//", linha)[0]
        return linha.strip()

    @staticmethod
    def _eh_numero(token: str) -> bool:

        token_up = token.upper()
        try:
            if token_up.startswith("0X"):
                int(token_up, 16)
            elif token_up.endswith("H") and len(token_up) > 1:
                int(token_up[:-1], 16)
            elif token_up.startswith("0B"):
                int(token_up, 2)
            else:
                int(token_up, 10)
            return True
        except ValueError:
            return False

    @classmethod
    def _converter_numero(cls, token: str, num_linha: int, eh_endereco: bool = False) -> int:
        """
        Converte string numérica para inteiro de 8 bits (0 a 255).
        Para dados, aceita valores negativos em complemento de 2 (-128 a 255).
        """
        token_up = token.upper()
        try:
            if token_up.startswith("0X"):
                valor = int(token_up, 16)
            elif token_up.endswith("H") and len(token_up) > 1:
                valor = int(token_up[:-1], 16)
            elif token_up.startswith("0B"):
                valor = int(token_up, 2)
            else:
                valor = int(token_up, 10)
        except ValueError:
            raise ErroDeMontagem(f"Linha {num_linha}: Valor numérico inválido '{token}'.")

        if eh_endereco:
            if not (0 <= valor < cls.TAMANHO_MEMORIA):
                raise ErroDeMontagem(
                    f"Linha {num_linha}: Endereço {valor} fora do limite da memória (0 a 255)."
                )
            return valor
        if not (-128 <= valor <= 255):
            raise ErroDeMontagem(
                f"Linha {num_linha}: Valor '{valor}' fora do limite de 8 bits (-128 a 255)."
            )
        return valor & 0xFF

    def _escrever_memoria(self, endereco: int, valor: int, num_linha: int, descricao: str = "") -> None:
        """Grava um byte validando limite de endereçamento e sobreposição."""
        if not (0 <= endereco < self.TAMANHO_MEMORIA):
            raise ErroDeMontagem(
                f"Linha {num_linha}: Estouro de memória ao tentar gravar no endereço {endereco}."
            )
        if endereco in self._escritos:
            raise ErroDeMontagem(
                f"Linha {num_linha}: Sobreposição, o endereço {endereco} já foi preenchido "
                f"(código e dados não podem ocupar o mesmo byte)."
            )
        self._escritos.add(endereco)
        self.memoria[endereco] = valor & 0xFF
        if descricao:
            self.log_montagem.append(
                f"End {endereco:03d} (0x{endereco:02X}) <- {self.memoria[endereco]:03d} "
                f"(0x{self.memoria[endereco]:02X}) | {descricao}"
            )

    # entrada 1: TXT do simulador
    def montar_txt_simulador(self, texto: str) -> List[int]:

        self.resetar()
        for idx, linha in enumerate(texto.splitlines(), start=1):
            tokens = linha.split()
            if not tokens:
                continue
            if len(tokens) < 2 or not tokens[0].isdigit() or not re.fullmatch(r"-?\d+", tokens[1]):
                raise ErroDeMontagem(f"Linha {idx}: Formato inválido '{linha.strip()}' "
                                     f"(esperado: endereço valor [operando] [mnemônico]).")
            endereco = self._converter_numero(tokens[0], idx, eh_endereco=True)
            valor = self._converter_numero(tokens[1], idx)
            self._escrever_memoria(endereco, valor, idx, f"Valor {tokens[1]}")
            if len(tokens) >= 3 and tokens[2].isdigit():
                operando = self._converter_numero(tokens[2], idx, eh_endereco=True)
                self._escrever_memoria(endereco + 1, operando, idx, f"Operando {tokens[2]}")
        return list(self.memoria)

    # entrada 2: Assembly
    def montar_texto(self, codigo_fonte: str) -> List[int]:

        self.resetar()

        # coleta de rótulos e constantes (EQU)
        pc_virtual = 0
        itens: List[Tuple[int, List[str]]] = []

        for idx, linha_bruta in enumerate(codigo_fonte.splitlines(), start=1):
            linha = self._limpar_linha(linha_bruta).replace("(", " ").replace(")", " ")
            if not linha.strip():
                continue

            if ":" in linha:
                nome, linha = linha.split(":", 1)
                nome = nome.strip().upper()
                if (not re.match(r"^[A-Z_][A-Z0-9_]*$", nome)
                        or nome in self.TABELA_INSTRUCOES or nome in self.DIRETIVAS):
                    raise ErroDeMontagem(f"Linha {idx}: Nome de rótulo inválido ou reservado '{nome}'.")
                if nome in self.tabela_simbolos:
                    raise ErroDeMontagem(f"Linha {idx}: Rótulo '{nome}' definido mais de uma vez.")
                self.tabela_simbolos[nome] = pc_virtual
                if not linha.strip():
                    continue

            tokens = [t for t in re.split(r"[\s,]+", linha) if t]
            if not tokens:
                continue
            primeiro = tokens[0].upper()

            if len(tokens) == 3 and tokens[1].upper() == "EQU":
                simbolo = tokens[0].upper()
                if simbolo in self.tabela_simbolos:
                    raise ErroDeMontagem(f"Linha {idx}: Símbolo '{simbolo}' definido mais de uma vez.")
                self.tabela_simbolos[simbolo] = self._converter_numero(tokens[2], idx)
                continue

            if primeiro == "ORG":
                if len(tokens) != 2:
                    raise ErroDeMontagem(f"Linha {idx}: Diretiva ORG exige um endereço (ex: ORG 128).")
                pc_virtual = self._converter_numero(tokens[1], idx, eh_endereco=True)
            elif primeiro == "DB":
                if len(tokens) < 2:
                    raise ErroDeMontagem(f"Linha {idx}: Diretiva DB exige ao menos um valor.")
                pc_virtual += len(tokens) - 1
            elif primeiro in self.TABELA_INSTRUCOES:
                pc_virtual += self.TABELA_INSTRUCOES[primeiro][1]
            else:
                raise ErroDeMontagem(f"Linha {idx}: Mnemônico ou comando desconhecido '{tokens[0]}'.")

            itens.append((idx, tokens))

        # geração do código de máquina
        pc_atual = 0
        for idx, tokens in itens:
            primeiro = tokens[0].upper()

            if primeiro == "ORG":
                pc_atual = self._converter_numero(tokens[1], idx, eh_endereco=True)

            elif primeiro == "DB":
                for token_dado in tokens[1:]:
                    valor = self._resolver_operando(token_dado, idx, permitir_negativo=True)
                    self._escrever_memoria(pc_atual, valor, idx, f"DB {token_dado}")
                    pc_atual += 1

            else:  # mnemônico
                opcode, tamanho = self.TABELA_INSTRUCOES[primeiro]
                if tamanho == 1 and len(tokens) != 1:
                    raise ErroDeMontagem(f"Linha {idx}: A instrução '{primeiro}' não possui operandos.")
                if tamanho == 2 and len(tokens) != 2:
                    raise ErroDeMontagem(
                        f"Linha {idx}: A instrução '{primeiro}' exige exatamente 1 operando (endereço)."
                    )
                self._escrever_memoria(pc_atual, opcode, idx, f"Instrução {primeiro}")
                pc_atual += 1
                if tamanho == 2:
                    operando = self._resolver_operando(tokens[1], idx)
                    self._escrever_memoria(pc_atual, operando, idx, f"Operando de {primeiro} ({tokens[1]})")
                    pc_atual += 1

        return list(self.memoria)

    def _resolver_operando(self, token: str, num_linha: int, permitir_negativo: bool = False) -> int:
        token_up = token.upper()
        if token_up in self.tabela_simbolos:
            return self.tabela_simbolos[token_up]
        if self._eh_numero(token):
            return self._converter_numero(token, num_linha, eh_endereco=not permitir_negativo)
        raise ErroDeMontagem(f"Linha {num_linha}: Símbolo ou endereço '{token}' não foi definido.")

    # detecção e entrada geral
    @staticmethod
    def detectar_formato(texto: str) -> str:
        for linha in texto.splitlines():
            if linha.strip():
                return "txt_simulador" if linha.split()[0].isdigit() else "assembly"
        return "assembly"

    def montar(self, texto: str) -> List[int]:
        if self.detectar_formato(texto) == "txt_simulador":
            return self.montar_txt_simulador(texto)
        return self.montar_texto(texto)

    def montar_arquivo(self, caminho_entrada: str) -> List[int]:
        if not os.path.exists(caminho_entrada):
            raise FileNotFoundError(f"Arquivo de entrada não encontrado: '{caminho_entrada}'")
        try:
            with open(caminho_entrada, "r", encoding="utf-8") as f:
                conteudo = f.read()
        except UnicodeDecodeError:
            with open(caminho_entrada, "r", encoding="latin-1") as f:
                conteudo = f.read()
        return self.montar(conteudo)

    # desmontador
    def desmontar(self, memoria: Optional[List[int]] = None) -> List[str]:

        mem = self.memoria if memoria is None else memoria
        linhas: List[str] = []
        i = 0
        while i < self.TAMANHO_MEMORIA:
            mnemonico = self.OPCODE_PARA_MNEMONICO.get(mem[i])
            if mnemonico is None:
                i += 1
                continue
            tamanho = self.TABELA_INSTRUCOES[mnemonico][1]
            if tamanho == 1:
                linhas.append(f"l{i}:  {mnemonico}")
            else:
                if i + 1 >= self.TAMANHO_MEMORIA:
                    break
                operando = mem[i + 1]
                alvo = f"l{operando}" if mnemonico.startswith("J") else str(operando)
                linhas.append(f"l{i}:  {mnemonico}({alvo})")
            i += tamanho
            if mnemonico == "HLT":
                break
        return linhas

    # saídas
    def salvar_mem_txt(self, caminho_saida: str) -> None:
        with open(caminho_saida, "w", encoding="utf-8") as f:
            for valor in self.memoria:
                f.write(f"{valor}\n")

    def salvar_st_h(self, caminho_saida: str) -> None:
        with open(caminho_saida, "w", encoding="utf-8") as f:
            f.write("\n".join(self.desmontar()))

    def salvar_hex(self, caminho_saida: str) -> None:
        with open(caminho_saida, "w", encoding="utf-8") as f:
            f.write("# ENDERECO : HEX  (DEC | BIN)\n")
            for end, val in enumerate(self.memoria):
                f.write(f"{end:03d} (0x{end:02X}) : 0x{val:02X} ({val:03d} | {val:08b})\n")

    def exibir_resumo_montagem(self) -> None:
        print("=" * 65)
        print("RESUMO DA MONTAGEM (MONTADOR AHMES)")
        print("=" * 65)
        if self.tabela_simbolos:
            print("Tabela de Símbolos (Rótulos/Constantes):", self.tabela_simbolos)
            print("-" * 65)
        for item in self.log_montagem:
            print(item)
        print("=" * 65)

## 4. Interface, I/O e Validação (Testes)

**Escopo:**
- Visualização legível do estado interno do simulador (registradores, flags e dump da memória útil).
- Código de integração que unirá as partes anteriores e executará testes práticos (como o programa Soma11).

In [40]:
def dump_memoria(cpu, titulo="Memória"):
    print(f"\n {titulo}:")
    for i in range(32):
        print(" ".join(f"{i + 32 * c}:{cpu.memoria[i + 32 * c]}" for c in range(8)))


def exibir_estado(cpu, mostrar_memoria=True):
    estado = cpu.estado_atual()

    print("=" * 45)
    print(f"CICLO {estado['ciclo']}")
    print("=" * 45)

    # Registradores
    print(f"PC: {estado['PC']}")
    print(f"AC: {estado['AC']}")

    # Flags
    print("\nFLAGS")
    print(" ".join(f"{nome}: {estado['Flags'][nome]}" for nome in ["N", "Z", "C", "V", "B"]))
    print(f"AC: {estado['Sinal']} | {estado['Zero']}")

    # Status da CPU
    print("\nStatus:", "Finalizado" if estado["Halted"] else "Executando")

    # Memória
    if mostrar_memoria:
        print("\nMEMÓRIA ATIVA")
        print("-" * 45)
        print("END | DEC | HEX | BINÁRIO")
        print("-" * 45)
        for endereco, valor in enumerate(cpu.memoria):
            if valor != 0:
                print(f"{endereco:03d} | {valor:03d} | 0x{valor:02X} | {valor:08b}")
        print("-" * 45)

In [41]:
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

In [42]:
import ipywidgets as widgets

SOMA11_ASM = """; Soma11 - soma 11+10+...+1 (resultado = 66 no endereço 201)
        LDA ZERO
        STA SOMA
        LDA N
LOOP:   STA CONT
        LDA SOMA
        ADD CONT
        STA SOMA
        LDA CONT
        SUB UM
        JNZ LOOP
        HLT
        ORG 200
ZERO:   DB 0
SOMA:   DB 0
CONT:   DB 0
N:      DB 11
UM:     DB 1
"""

cabecalho = widgets.HTML(value="""
  <div style="background-color:#10141E;padding:20px;border-radius:10px;font-family:Arial;">
    <h1 style="color:#60A5FA;margin:0;">AHMES</h1>
    <p style="color:#94A3B8;margin-bottom:0;">Simulador de Arquitetura Computacional de 8 Bits</p>
  </div>
""")

CARD = "background:#1C2432;padding:14px;border-radius:10px;flex:1;text-align:center;"

painel_registradores = widgets.HTML()
painel_memoria = widgets.HTML()
saida = widgets.Output()


def atualizar_registradores(cpu):
    e = cpu.estado_atual()
    flags = "".join(
        f"<span style='display:inline-block;margin:0 4px;padding:4px 10px;border-radius:6px;"
        f"background:{'#16A34A' if e['Flags'][n] else '#334155'};color:#fff;'>{n}={e['Flags'][n]}</span>"
        for n in ["N", "Z", "C", "V", "B"]
    )
    situacao = "Finalizado" if e["Halted"] else "Pronto"
    painel_registradores.value = f"""
    <div style="display:flex;gap:10px;font-family:Arial;">
      <div style="{CARD}"><span style="color:#94A3B8;">PC</span>
        <h2 style="color:#60A5FA;margin:4px 0;">{e['PC']}</h2></div>
      <div style="{CARD}"><span style="color:#94A3B8;">AC</span>
        <h2 style="color:#60A5FA;margin:4px 0;">{e['AC']}</h2>
        <span style="color:#94A3B8;">{e['Sinal']} | {e['Zero']}</span></div>
      <div style="{CARD}"><span style="color:#94A3B8;">Ciclo</span>
        <h2 style="color:#60A5FA;margin:4px 0;">{e['ciclo']}</h2>
        <span style="color:#94A3B8;">{situacao}</span></div>
    </div>
    <div style="font-family:Arial;margin-top:10px;">{flags}</div>
    """



def atualizar_memoria(cpu):
    linhas = ""
    for end, v in enumerate(cpu.memoria):
        if end == cpu.pc and not cpu.halted:
            cor = "background:#1E3A5F;"
        elif v != memoria_carregada[end]:
            cor = "background:#14532D;"
        else:
            cor = ""

        linhas += f"""
            <tr style="{cor}">
                <td>{end:03d}</td>
                <td>{v:03d}</td>
                <td>0x{v:02X}</td>
                <td>{v:08b}</td>
            </tr>
        """

    painel_memoria.value = f"""
        <div style="
            font-family:monospace;
            color:#E2E8F0;
            background:#10141E;
            padding:15px;
            border-radius:10px;
        ">
            <h3>MEMÓRIA RAM — 256 BYTES</h3>
            <p style="color:#60A5FA;">
                Azul: próxima instrução (PC)
            </p>
            <p style="color:#4ADE80;">
                Verde: valor diferente da memória inicial
            </p>
            <div style="max-height:350px;overflow-y:auto;">
                <table style="width:100%;text-align:center;">
                    <tr style="color:#94A3B8;">
                        <th>ENDEREÇO</th>
                        <th>DECIMAL</th>
                        <th>HEX</th>
                        <th>BINÁRIO</th>
                    </tr>
                    {linhas}
                </table>
            </div>
        </div>
    """

def atualizar_tudo():
    atualizar_registradores(cpu_interface)
    atualizar_memoria(cpu_interface)


In [43]:
# Estado da interface
cpu_interface = AhmesCore(ula_executor=ULA_EM_USO)
memoria_carregada = [0] * 256

entrada_programa = widgets.Textarea(
    value=SOMA11_ASM,
    description="Programa:",
    layout=widgets.Layout(width="98%", height="170px"),
    style={"description_width": "initial"},
)

botao_carregar = widgets.Button(description="Montar e carregar", button_style="success", icon="upload")
botao_passo    = widgets.Button(description="Executar passo", button_style="primary", icon="step-forward")
botao_tudo     = widgets.Button(description="Executar tudo", button_style="info", icon="play")
botao_reset    = widgets.Button(description="Reset", button_style="warning", icon="refresh")
botao_salvar   = widgets.Button(description="Gerar mem.txt e ST.h", icon="download")


def _restaurar_cpu():
    cpu_interface.reset()
    cpu_interface.limpar_memoria()
    cpu_interface.carregar_programa(memoria_carregada)


def _mostrar_erro(erro):
    with saida:
        print(f"[ERRO] {type(erro).__name__}: {erro}")


def ao_carregar(_):
    global memoria_carregada
    saida.clear_output()
    montador = MontadorAhmes()
    try:
        memoria_carregada = montador.montar(entrada_programa.value)
    except ErroDeMontagem as erro:
        _mostrar_erro(erro)
        return
    _restaurar_cpu()
    atualizar_tudo()
    with saida:
        print(f"Montagem concluída ({montador.detectar_formato(entrada_programa.value)}).")
        if montador.tabela_simbolos:
            print("Símbolos:", montador.tabela_simbolos)


def ao_passo(_):
    saida.clear_output()
    try:
        cpu_interface.executar_passo()
    except Exception as erro:
        _mostrar_erro(erro)
    atualizar_tudo()


def ao_executar_tudo(_):
    saida.clear_output()
    try:
        cpu_interface.executar_ate_parar()
    except Exception as erro:
        _mostrar_erro(erro)
    atualizar_tudo()


def ao_reset(_):
    saida.clear_output()
    _restaurar_cpu()
    atualizar_tudo()


def ao_salvar(_):
    saida.clear_output()
    montador = MontadorAhmes()
    try:
        montador.montar(entrada_programa.value)
        montador.salvar_mem_txt("mem.txt")
        montador.salvar_st_h("ST.h")
    except ErroDeMontagem as erro:
        _mostrar_erro(erro)
        return
    with saida:
        print("Arquivos gerados: mem.txt e ST.h")
        try:
            from google.colab import files
            files.download("mem.txt")
            files.download("ST.h")
        except Exception:
            pass


botao_carregar.on_click(ao_carregar)
botao_passo.on_click(ao_passo)
botao_tudo.on_click(ao_executar_tudo)
botao_reset.on_click(ao_reset)
botao_salvar.on_click(ao_salvar)


## Rodar tudo


In [44]:

from IPython.display import display

ao_carregar(None)   # já monta e carrega o Soma11 ao abrir
display(widgets.VBox([
    cabecalho,
    painel_registradores,
    entrada_programa,
    widgets.HBox([botao_carregar, botao_passo, botao_tudo, botao_reset, botao_salvar]),
    saida,
    painel_memoria,
]))
